# Part 1: Feature Engineering for Customer Churn Prediction

Welcome to the lab. In this exercise, you'll transform raw customer data into a clean, machine-learning-ready dataset.

## Your Objective
Prepare TeleConnect's customer data so it can be used to predict customer churn.

## How to Use This Notebook
1. Read the text in each section carefully
2. Run the code cells by clicking the 'run' button in the top left of each cell, or by pressing Shift + Enter in the cell 
3. Observe the outputs and visualizations
4. Complete the "YOUR TURN" sections by running code and experimenting with values
5. Discuss with your group what the outputs mean for the business

**Important:** All code is provided for you. Your job is to run it, understand it, and experiment with it.

Let's begin.

---
## Section 1: Setup & Data Loading

First, we need to import the Python libraries we'll use:
- pandas: For working with data tables (like Excel, but more powerful)
- numpy: For mathematical operations
- matplotlib & seaborn: For creating charts and visualizations

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

### Load the Customer Data

Now let's load our CSV file into a pandas DataFrame.

In [ ]:
# Load the dataset
df = pd.read_csv('customer_data.csv')

print(f"Data loaded successfully!")
print(f"Number of customers: {len(df)}")
print(f"Number of features: {len(df.columns)}")
print(f"\nColumn names:")
print(df.columns.tolist())

### First Look at the Data

Let's examine the first few rows to understand what we're working with.

In [ ]:
# Display the first 5 rows
df.head()

### Data Profiling: Understanding Our Dataset

Before cleaning data, we need to understand it. Let's look at:
- Data types (numbers vs text)
- Basic statistics
- Missing values

In [ ]:
# Check data types and missing values
print("Dataset Info:")
print("=" * 50)
df.info()

In [ ]:
# Statistical summary for numerical columns
print("Statistical Summary:")
print("=" * 50)
df.describe()

### Visualize the Target Variable: Churn

Let's see how many customers churned vs stayed. This is what we're trying to predict.

In [ ]:
# Count and visualize churn distribution
churn_counts = df['Churned'].value_counts()
churn_percentages = df['Churned'].value_counts(normalize=True) * 100

print("Churn Distribution:")
print(f"  Stayed (0): {churn_counts[0]} customers ({churn_percentages[0]:.1f}%)")
print(f"  Churned (1): {churn_counts[1]} customers ({churn_percentages[1]:.1f}%)")

# Create visualization
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
churn_counts.plot(kind='bar', ax=ax[0], color=['#2ecc71', '#e74c3c'])
ax[0].set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')
ax[0].set_xlabel('Churned (0=No, 1=Yes)')
ax[0].set_ylabel('Number of Customers')
ax[0].set_xticklabels(['Stayed', 'Churned'], rotation=0)

# Pie chart
ax[1].pie(churn_counts, labels=['Stayed', 'Churned'], autopct='%1.1f%%', 
          colors=['#2ecc71', '#e74c3c'], startangle=90)
ax[1].set_title('Churn Rate', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nBusiness Insight: {churn_percentages[1]:.1f}% of customers have churned.")
print(f"If we can predict and prevent even half of these, we could retain ~{churn_counts[1]//2} customers.")

---
## Section 2: Handling Missing Data

Missing data is one of the most common problems in real-world datasets. Let's identify where data is missing and fix it.

### Why Data Goes Missing:
- MCAR (Missing Completely At Random): Random errors, like a sensor failing
- MAR (Missing At Random): Missing data correlated with other variables
- MNAR (Missing Not At Random): The missingness is related to the value itself

In [ ]:
# Identify missing values
missing_data = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_data,
    'Percentage': missing_percent
})

missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

print("Missing Data Summary:")
print("=" * 50)
print(missing_summary)
print(f"\nTotal missing values: {missing_data.sum()}")

### Visualize Missing Data Pattern

In [ ]:
# Create a heatmap showing missing data
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Data Pattern (Yellow = Missing)', fontsize=14, fontweight='bold')
plt.xlabel('Features')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Fix Missing Data: Age (MCAR Example)

Age is missing randomly. For numerical data, we can use the median (middle value) to fill in gaps.

Why median instead of mean? It's not affected by extreme values (outliers).

In [ ]:
# Check Age before imputation
print(f"Age - Missing values BEFORE: {df['Age'].isnull().sum()}")
print(f"Age - Median value: {df['Age'].median():.0f} years")

# Impute missing Age with median
df['Age'].fillna(df['Age'].median(), inplace=True)

print(f"Age - Missing values AFTER: {df['Age'].isnull().sum()}")
print("Age missing values filled with median!")

### Fix Missing MonthlyCharges (MAR Example)

MonthlyCharges is missing more often for Month-to-Month customers. Let's fill it with the mean charges for that customer's contract type.

This is called "group-based imputation" because we use the average for each group (contract type) rather than a single overall average.

### Fix Missing SupportCalls (MNAR Example)

Support calls are missing when customers made many calls (system couldn't keep up). Since these are likely high values, we'll fill with median for now (conservative approach).

Alternative approaches:
- Fill with max value (aggressive)
- Create a "missing" indicator flag
- Use predictive imputation

In [ ]:
# Check distribution of SupportCalls first
print(f"SupportCalls - Missing values: {df['SupportCalls'].isnull().sum()}")
print(f"SupportCalls - Mean (non-missing): {df['SupportCalls'].mean():.1f}")
print(f"SupportCalls - Max (non-missing): {df['SupportCalls'].max():.0f}")

# For now, let's create a flag to remember these were missing
# This preserves information that might be useful for prediction
df['SupportCalls_WasMissing'] = df['SupportCalls'].isnull().astype(int)

# Fill with median value
df['SupportCalls'].fillna(df['SupportCalls'].median(), inplace=True)

print(f"\nSupportCalls - Missing values AFTER: {df['SupportCalls'].isnull().sum()}")
print(f"Created 'SupportCalls_WasMissing' flag and imputed values!")
print(f"{df['SupportCalls_WasMissing'].sum()} customers had missing support call data")

### Check: All Missing Data Resolved?

In [ ]:
# Verify no missing data remains
remaining_missing = df.isnull().sum().sum()

if remaining_missing == 0:
    print("SUCCESS! No missing data remaining!")
    print(f"Dataset is now {len(df)} rows x {len(df.columns)} columns")
else:
    print(f"Warning: {remaining_missing} missing values still present")
    print(df.isnull().sum())

---
## Section 3: Data Cleaning

Now let's clean up messy data:
1. Remove duplicate records
2. Standardize inconsistent values
3. Handle outliers

### 1. Remove Duplicates

Duplicate records can skew our analysis and model training.

In [ ]:
# Check for duplicates
duplicate_count = df.duplicated().sum()
print(f"Found {duplicate_count} duplicate rows")

if duplicate_count > 0:
    # Show example of duplicates
    print("\nExample duplicates:")
    print(df[df.duplicated(keep=False)].head(4))
    
    # Remove duplicates
    df_before = len(df)
    df = df.drop_duplicates()
    df_after = len(df)
    
    print(f"\nRemoved {df_before - df_after} duplicate rows")
    print(f"Dataset now has {df_after} unique customers")
else:
    print("No duplicates found!")

### 2. Standardize PhoneService Values

Notice that PhoneService has inconsistent values: 'Yes', 'YES', 'yes', 'Y', etc.
Let's standardize them!

In [ ]:
# Check current values
print("PhoneService values BEFORE standardization:")
print(df['PhoneService'].value_counts())

# Standardize to 'Yes' or 'No'
# Map all variations of 'yes' to 'Yes', all variations of 'no' to 'No'
yes_variations = ['YES', 'yes', 'Y', 'y']
no_variations = ['NO', 'no', 'N', 'n']

df['PhoneService'] = df['PhoneService'].replace(yes_variations, 'Yes')
df['PhoneService'] = df['PhoneService'].replace(no_variations, 'No')

print("\nPhoneService values AFTER standardization:")
print(df['PhoneService'].value_counts())
print("\nPhoneService values standardized!")

### 3. Handle Outliers in MonthlyCharges

Outliers are extreme values that might be errors or special cases. Let's visualize them.

In [ ]:
# Visualize MonthlyCharges distribution with boxplot
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
ax[0].boxplot(df['MonthlyCharges'])
ax[0].set_title('MonthlyCharges - Boxplot', fontsize=12, fontweight='bold')
ax[0].set_ylabel('Monthly Charges ($)')
ax[0].grid(True, alpha=0.3)

# Histogram
ax[1].hist(df['MonthlyCharges'], bins=30, edgecolor='black', alpha=0.7)
ax[1].set_title('MonthlyCharges - Distribution', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Monthly Charges ($)')
ax[1].set_ylabel('Number of Customers')
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate outliers using IQR method
Q1 = df['MonthlyCharges'].quantile(0.25)
Q3 = df['MonthlyCharges'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['MonthlyCharges'] < lower_bound) | (df['MonthlyCharges'] > upper_bound)]

print(f"MonthlyCharges Statistics:")
print(f"   Mean: ${df['MonthlyCharges'].mean():.2f}")
print(f"   Median: ${df['MonthlyCharges'].median():.2f}")
print(f"   Range: ${df['MonthlyCharges'].min():.2f} - ${df['MonthlyCharges'].max():.2f}")
print(f"\nOutliers detected: {len(outliers)} customers ({len(outliers)/len(df)*100:.1f}%)")
print(f"\nBusiness Decision: Keep outliers (they represent real high-value/low-value customers)")

---
## Section 4: Feature Engineering

Now the fun part! We'll transform our data to make it useful for machine learning.

### What is Feature Engineering?
Creating or transforming features (columns) to help the model learn patterns better.

### 1. Create New Features

Sometimes combining existing features creates powerful new insights!

In [ ]:
# Create new calculated features

# 1. Average charges per month of tenure
df['AvgChargesPerMonth'] = df['TotalCharges'] / df['TenureMonths']

# 2. Is customer a senior? (Age >= 65)
df['IsSenior'] = (df['Age'] >= 65).astype(int)

# 3. Is customer new? (Less than 6 months)
df['IsNewCustomer'] = (df['TenureMonths'] < 6).astype(int)

# 4. High support needs? (More than 3 calls)
df['HighSupportNeeds'] = (df['SupportCalls'] > 3).astype(int)

print("Created 4 new features:")
print("   - AvgChargesPerMonth")
print("   - IsSenior")
print("   - IsNewCustomer")
print("   - HighSupportNeeds")

print(f"\nSample of new features:")
print(df[['Age', 'IsSenior', 'TenureMonths', 'IsNewCustomer', 'SupportCalls', 'HighSupportNeeds']].head())

### 2. Binning: Convert Age into Age Groups

Binning groups continuous numbers into categories. This can help models find patterns.

Example: Instead of exact ages (22, 23, 24...), use groups (18-30, 31-45, 46-60, 60+)

In [ ]:
# Create age bins
bins = [0, 30, 45, 60, 100]
labels = ['18-30', '31-45', '46-60', '60+']

df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

print("Created AgeGroup feature!")
print("\nDistribution of customers by age group:")
print(df['AgeGroup'].value_counts().sort_index())

# Visualize
plt.figure(figsize=(10, 5))
df['AgeGroup'].value_counts().sort_index().plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Customer Distribution by Age Group', fontsize=14, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### YOUR TURN: Create Tenure Groups (Run & Experiment)

Now it's your turn to create bins! We'll group customers by how long they've been with the company.

**What to do:**
1. Run the code cell below to create the TenureGroup feature
2. Look at the distribution and visualization
3. EXPERIMENT: Try changing the numbers in `tenure_bins` to see how it affects the groups
   - Example: Change `[0, 12, 24, 48, 100]` to `[0, 6, 24, 36, 100]` to make different groups
   - This shows you how business decisions (what counts as "new" vs "loyal") affect your analysis

**Current groups:**
- 0-12 months: "New"
- 13-24 months: "Medium"  
- 25-48 months: "Established"
- 48+ months: "Loyal"

In [ ]:
# Create tenure bins - these numbers define the group boundaries
tenure_bins = [0, 12, 24, 48, 100]
tenure_labels = ['New', 'Medium', 'Established', 'Loyal']

# Apply the binning
df['TenureGroup'] = pd.cut(df['TenureMonths'], bins=tenure_bins, labels=tenure_labels)

# Check the results
print("Created TenureGroup feature!")
print("\nDistribution of customers by tenure:")
print(df['TenureGroup'].value_counts().sort_index())

# Visualize
plt.figure(figsize=(10, 5))
df['TenureGroup'].value_counts().sort_index().plot(kind='bar', color='lightgreen', edgecolor='black')
plt.title('Customer Distribution by Tenure Group', fontsize=14, fontweight='bold')
plt.xlabel('Tenure Group')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3. One-Hot Encoding: Converting Categories to Numbers

The Problem: Machine learning models only understand numbers, not text like "Fiber Optic" or "DSL".

The Solution: One-hot encoding creates separate yes/no (1/0) columns for each category.

Example:
```
ContractType          →    ContractType_Month-to-Month  ContractType_OneYear  ContractType_TwoYear
Month-to-Month       →              1                          0                      0
One Year              →              0                          1                      0
```

In [ ]:
# Let's see which columns are categorical (text or category dtype)
categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Remove CustomerID (we don't want to encode that!)
if 'CustomerID' in categorical_columns:
    categorical_columns.remove('CustomerID')

print("Categorical columns to encode:")
for col in categorical_columns:
    print(f"   - {col}: {df[col].nunique()} unique values")

In [ ]:
# Perform one-hot encoding
print(f"Columns BEFORE encoding: {len(df.columns)}")

# Use pandas get_dummies() for one-hot encoding
df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=True, dtype=int)

# Remove CustomerID - we don't need it for modeling
df_encoded = df_encoded.drop('CustomerID', axis=1, errors='ignore')

print(f"Columns AFTER encoding: {len(df_encoded.columns)}")
print(f"Added {len(df_encoded.columns) - len(df.columns)} new columns!")

print("\nSample of encoded columns:")
encoded_cols = [col for col in df_encoded.columns if col not in df.columns]
print(encoded_cols[:10])  # Show first 10 new columns

### Why drop_first=True?

This avoids the "dummy variable trap" - redundancy where one category can be inferred from others.

Example: If ContractType_OneYear=0 and ContractType_TwoYear=0, we know it must be Month-to-Month!

In [ ]:
# Let's look at the encoded data
print("First few rows of encoded dataset:")
df_encoded.head()

### 4. Correlation Analysis

Correlation measures how strongly two variables are related (-1 to +1).
- +1: Perfect positive correlation (both increase together)
- 0: No correlation
- -1: Perfect negative correlation (one increases, other decreases)

Let's find which features are most correlated with Churn!

In [ ]:
# Calculate correlation with Churned
correlations = df_encoded.select_dtypes(include=[np.number]).corr()['Churned'].sort_values(ascending=False)

print("Top 15 Features Most Correlated with Churn:")
print("=" * 60)
print(correlations.head(15))

print("\nBottom 10 Features (Negatively Correlated with Churn):")
print("=" * 60)
print(correlations.tail(10))

In [ ]:
# Visualize top correlations
top_features = correlations.head(10).drop('Churned')  # Remove Churned itself

plt.figure(figsize=(10, 6))
top_features.plot(kind='barh', color='coral', edgecolor='black')
plt.title('Top 9 Features Correlated with Customer Churn', fontsize=14, fontweight='bold')
plt.xlabel('Correlation Coefficient')
plt.ylabel('Feature')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nBusiness Insights:")
print("   - Positive correlation: Feature increases → Churn increases")
print("   - Negative correlation: Feature increases → Churn decreases")

### Create a Full Correlation Heatmap

In [ ]:
# Select numerical features for correlation matrix
numerical_features = ['Age', 'TenureMonths', 'MonthlyCharges', 'TotalCharges', 
                      'SupportCalls', 'AvgChargesPerMonth', 'Churned']

plt.figure(figsize=(10, 8))
sns.heatmap(df_encoded[numerical_features].corr(), annot=True, fmt='.2f', 
            cmap='coolwarm', center=0, square=True, linewidths=1)
plt.title('Correlation Heatmap - Key Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Look for:")
print("   - Dark red: Strong positive correlation")
print("   - Dark blue: Strong negative correlation")
print("   - White: No correlation")

---
## Section 5: Train/Test Split

### Why Split Data?

Imagine studying with a practice test, then taking THE SAME TEST for your final exam. You'd get 100%, but did you really learn?

Same concept here:
- Training Set (80%): Model learns from this
- Testing Set (20%): Model proves it learned (never seen before!)

This prevents overfitting (memorizing instead of learning).

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features (X) from target (y)
# Features: Everything except CustomerID and Churned
X = df_encoded.drop(['CustomerID', 'Churned'], axis=1, errors='ignore')
y = df_encoded['Churned']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nNumber of features: {X.shape[1]}")
print(f"Number of samples: {X.shape[0]}")

In [ ]:
# Split into train and test sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20,      # 20% for testing
    random_state=438,    # For reproducibility
    stratify=y           # Keep same churn ratio in both sets
)

print("Data split completed!")
print("\nTraining Set:")
print(f"   Features: {X_train.shape}")
print(f"   Target: {y_train.shape}")
print(f"   Churn rate: {y_train.mean()*100:.1f}%")

print("\nTesting Set:")
print(f"   Features: {X_test.shape}")
print(f"   Target: {y_test.shape}")
print(f"   Churn rate: {y_test.mean()*100:.1f}%")

print("\nNotice: Churn rates are similar in both sets (thanks to stratify=y)")

### Save Processed Data

Let's save our cleaned and engineered data for Part 2.

In [ ]:
# Save the train and test sets
import pickle

# Save as pickle files (preserves data types)
with open('X_train.pkl', 'wb') as f:
    pickle.dump(X_train, f)
    
with open('X_test.pkl', 'wb') as f:
    pickle.dump(X_test, f)
    
with open('y_train.pkl', 'wb') as f:
    pickle.dump(y_train, f)
    
with open('y_test.pkl', 'wb') as f:
    pickle.dump(y_test, f)

print("Saved processed data:")
print("   - X_train.pkl")
print("   - X_test.pkl")
print("   - y_train.pkl")
print("   - y_test.pkl")
print("\nReady for Part 2: Model Training!")

---
## Congratulations!

You've completed Part 1: Feature Engineering.

### What You Accomplished:

1. Loaded and explored customer data
2. Handled missing data using multiple strategies (MCAR, MAR, MNAR)
3. Cleaned messy data (duplicates, inconsistent values)
4. Created new features to improve predictions
5. Used binning to group continuous variables
6. Applied one-hot encoding to convert categories to numbers
7. Analyzed correlations to find important features
8. Split data into training and testing sets

### Final Dataset Stats:
- Original: 1,215 rows × 12 columns (with missing data)
- Final: ~1,200 rows × 25+ columns (clean and ML-ready)
- Training set: 80% of data
- Testing set: 20% of data

### Business Value Created:
You've transformed raw, messy customer data into a high-quality dataset that can:
- Predict which customers will churn
- Identify key factors driving churn
- Enable targeted retention campaigns
- Potentially save TeleConnect millions in lost revenue

### Next Steps:
In Part 2, you'll:
- Train a machine learning model
- Make predictions on new customers
- Evaluate model performance
- Extract business insights

---

## Reflection Questions

Take a few minutes to discuss with your group:

1. Which step was most surprising to you? Why?

2. How would you explain one-hot encoding to a non-technical manager?

3. Looking at the correlation analysis, what would you recommend to reduce churn?

4. What real-world business problems could use similar techniques?

5. What ethical considerations should TeleConnect think about when using this model?

---

Great work! See you in Part 2.